In [1]:
import streamlit as st
import pandas as pd
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="901012",
    database="e_commerce"
)

query = """
SELECT
    p.data_pedido,
    pr.nome_produto,
    i.quantidade,
    i.preco_unitario,
    (i.quantidade * i.preco_unitario) AS valor_total
FROM pedidos_silver p
JOIN itens_pedido_silver i ON p.pedido_id = i.pedido_id
JOIN produtos_silver pr ON i.produto_id = pr.produto_id
"""

df = pd.read_sql(query, conn)
df["data_pedido"] = pd.to_datetime(df["data_pedido"])

C:\Users\lucas\AppData\Local\Temp\ipykernel_3292\1991064183.py:24: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [2]:
df["ano"] = df["data_pedido"].dt.year
df["mes"] = df["data_pedido"].dt.month

In [4]:
st.title("📊 Dashboard de Vendas")

ano_sel = st.selectbox(
    "Selecione o ano",
    sorted(df["ano"].unique())
)

mes_sel = st.selectbox(
    "Selecione o mês",
    sorted(df[df["ano"] == ano_sel]["mes"].unique())
)

produto_sel = st.multiselect(
    "Selecione o produto",
    df["nome_produto"].unique(),
    default=df["nome_produto"].unique())

2026-02-25 13:02:49.643 
  command:

    streamlit run C:\Users\lucas\AppData\Roaming\Python\Python312\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-02-25 13:02:49.644 Session state does not function when running a script without `streamlit run`


In [5]:
df_filtrado = df[
    (df["ano"] == ano_sel) &
    (df["mes"] == mes_sel) &
    (df["nome_produto"].isin(produto_sel))
]

In [6]:
st.subheader("Faturamento por Produto")

faturamento_produto = (
    df_filtrado
    .groupby("nome_produto")["valor_total"]
    .sum()
    .reset_index()
)

st.bar_chart(
    faturamento_produto.set_index("nome_produto")
)

DeltaGenerator()

In [7]:
import sys
import os

# Import do projeto
sys.path.append(os.path.abspath("../python"))
from db_connection import get_connection

st.set_page_config(page_title="Dashboard E-commerce", layout="wide")
st.title("📊 Dashboard E-commerce")

conn = get_connection()

# ---- Carregar dados da Gold
df_faturamento = pd.read_sql(
    "SELECT * FROM vw_faturamento_mensal_gold",
    conn
)

df_ticket = pd.read_sql(
    "SELECT * FROM vw_ticket_medio_gold",
    conn
)

df_top_produtos = pd.read_sql(
    "SELECT * FROM vw_top_produtos_gold",
    conn
)

# ---- KPIs
col1, col2, col3 = st.columns(3)

col1.metric(
    "💰 Faturamento Total",
    f"R$ {df_ticket['faturamento_total'][0]:,.2f}"
)

col2.metric(
    "🧾 Total de Pedidos",
    int(df_ticket['total_pedidos'][0])
)

col3.metric(
    "🎯 Ticket Médio",
    f"R$ {df_ticket['ticket_medio'][0]:,.2f}"
)

st.divider()

# ---- Gráfico de faturamento
st.subheader("📈 Faturamento Mensal")
st.line_chart(
    df_faturamento.set_index("mes")["faturamento"]
)

# ---- Tabela Top Produtos
st.subheader("🏆 Top Produtos")
st.dataframe(df_top_produtos.head(10))

C:\Users\lucas\AppData\Local\Temp\ipykernel_3292\3504701420.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_faturamento = pd.read_sql(
C:\Users\lucas\AppData\Local\Temp\ipykernel_3292\3504701420.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_ticket = pd.read_sql(
C:\Users\lucas\AppData\Local\Temp\ipykernel_3292\3504701420.py:24: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_top_produtos = pd.read_sql(


DeltaGenerator()

In [ ]:
pwe